# Site → Test Plan generator

Give it the URL of a web application and it writes a QA test plan for it.

1. **The LLM picks the pages worth testing.** It gets every link on the landing page and returns JSON with the key user flows (login, search, cart, checkout...) and a priority for each.
2. **The code reads those pages**, including their forms and input fields.
3. **The LLM writes the test plan**: test cases per flow, edge cases and risks.

Target site: [automationexercise.com](https://automationexercise.com), a demo shop made for test automation practice.
Change `SITE_URL` to try another site. It must render its HTML on the server: single-page JavaScript apps return an almost empty page to `requests`, so no links are found.

In [ ]:
# Imports and settings

import json
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
import requests

from site_reader import get_links, get_page_text, load_page, read_page_for_testing

load_dotenv(override=True)
openai = OpenAI()

MODEL = "gpt-6-luna"
SITE_URL = "https://automationexercise.com"
MAX_PAGES = 6            # how many flow pages to read
PROMPT_LIMIT = 15_000    # characters sent to the LLM in step 3

PLANS_DIR = Path("test_plans")
PLANS_DIR.mkdir(exist_ok=True)

## Step 1: the LLM picks the pages worth testing

The model gets the list of links and answers in JSON, so the code can loop over its choices.
Besides filtering, it gives each page a `priority`, and the pages are sorted by it.
Anchors (`#...`), `mailto:` links and duplicates are removed first to keep the prompt short.

In [ ]:
link_system_prompt = """
You are a senior QA engineer planning testing for a web application.
You get the list of links found on its landing page.
Pick the pages that hold the key user flows worth testing, for example:
login / sign-up, search, product listing, product details, cart, checkout,
account pages, contact forms, file uploads.

Rules:
- When many links follow the same pattern (e.g. /product_details/1, /product_details/2),
  pick only ONE of them as a representative.
- Skip social media, legal pages, plain marketing content and external sites.
- Turn relative links into full https URLs.
- priority is "high" for flows that involve money, accounts or personal data,
  "medium" for core browsing, "low" for the rest.

Respond in JSON like this example:
{
    "flows": [
        {"flow": "authentication", "url": "https://shop.com/login", "priority": "high"},
        {"flow": "search", "url": "https://shop.com/search", "priority": "medium"}
    ]
}
"""

def get_links_user_prompt(url):
    return f"Here are the links on the web application {url}:\n\n" + "\n".join(get_links(url))


def select_flows_to_test(url):
    print(f"Asking {MODEL} which pages of {url} are worth testing...")
    response = openai.chat.completions.create(
        model=MODEL,
        reasoning_effort="none",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)},
        ],
        response_format={"type": "json_object"},
    )
    flows = json.loads(response.choices[0].message.content)["flows"]
    order = {"high": 0, "medium": 1, "low": 2}
    flows.sort(key=lambda f: order.get(f.get("priority"), 3))
    print(f"Found {len(flows)} flows")
    return flows

In [ ]:
flows = select_flows_to_test(SITE_URL)
for f in flows:
    print(f"{f['priority']:<7} {f['flow']:<25} {f['url']}")

## Step 2: read the chosen pages, including their forms

For each page, `read_page_for_testing` (in `site_reader.py`) keeps the visible text **and** a list of every form:
field names, types, and which fields are required. The forms are what make the test cases specific,
because that is where users type data and where most bugs show up.

In [ ]:
def collect_site_contents(url, flows):
    result = f"## Landing page\n\n{get_page_text(load_page(url))}\n"
    for flow in flows[:MAX_PAGES]:
        print(f"Reading {flow['flow']}: {flow['url']}")
        result += f"\n\n## Flow: {flow['flow']} (priority: {flow['priority']})\nURL: {flow['url']}\n\n"
        try:
            result += read_page_for_testing(flow["url"])
        except requests.RequestException as e:
            result += f"Could not load this page: {e}\n"
    return result

In [ ]:
site_contents = collect_site_contents(SITE_URL, flows)
print(f"\nCollected {len(site_contents):,} characters")
print(site_contents[:3000])

## Step 3: write the test plan

A second LLM call turns the collected pages into a test plan. The answer streams in as it is written
and is saved to `test_plans/`.

In [ ]:
test_plan_system_prompt = """
You are a senior QA engineer. From the page contents and forms of a web application,
write a test plan in markdown (no code blocks around it) with these sections:

1. **Scope** - the features you found, and what is out of scope
2. **Test cases** - one table per flow, highest priority flows first, columns:
   ID (e.g. AUTH-01) | Title | Steps | Expected result | Priority
   Cover the happy path AND negative / edge cases: empty required fields, invalid formats,
   special characters, very long input, duplicate data, quantity 0 or negative.
   Base the cases on the real field names you were given.
3. **Risks and open questions** - things you could not see from the page text
   (for example payment processing, emails, permissions).

Use only what is in the page contents. Do not invent features that are not there.
"""

def get_test_plan_user_prompt(url, site_contents):
    user_prompt = (
        f"You are planning testing for the web application at {url}.\n"
        "Here are its landing page and its key flow pages, with their forms:\n\n"
    )
    return (user_prompt + site_contents)[:PROMPT_LIMIT]


def stream_test_plan(url, site_contents):
    stream = openai.chat.completions.create(
        model=MODEL,
        reasoning_effort="none",
        messages=[
            {"role": "system", "content": test_plan_system_prompt},
            {"role": "user", "content": get_test_plan_user_prompt(url, site_contents)},
        ],
        stream=True,
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        update_display(Markdown(response), display_id=display_handle.display_id)
    return response

In [ ]:
plan = stream_test_plan(SITE_URL, site_contents)

site_name = SITE_URL.split("//")[1].split("/")[0].replace("www.", "").split(".")[0]
plan_file = PLANS_DIR / f"{site_name}.md"
plan_file.write_text(plan, encoding="utf-8")
print(f"Saved to {plan_file}")